# 05. Prospective Linebuster Bottleneck, Channel Audit & Causal ROI Study (Mon–Fri 07:00–10:00 AM)

**Objective:** Build an auditable ex-ante business case scoped specifically to the proposed **Mon–Fri 07:00–10:00 AM** (`target_hours=(7, 8, 9)`, `~21.7 weekdays/mo`, `~65.1 hrs/mo`) linebuster shift (`LINEBUSTER_START_DATE = None`), alongside full-store comparisons, bar production capacity (`tickets/hr`, `drinks/hr`), attach-rate dilution stress tests, and scoped MDE pilot power sizing.

**Mode Contract:** Reads `data/processed/incidents_raw.csv` in `raw` mode (never `data/sample/incidents.csv`).

## 1. Environment Setup & Causal ROI Imports

**Objective:** Configure PID-isolated matplotlib environment, HTML/plot helpers, `StoreConfig`, and `src.causal_eval` estimators.

**Inputs / Assumptions:** Reads ROI placeholders (`hardware_capex`, `monthly_saas_fee`, `peak_labor_cost_per_hr`, `gross_margin_pct`) from `StoreConfig`.

**How to Read the Output:** Confirms active data mode, `linebuster_start_date`, and `tablet_channel_interpretation`.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.causal_eval import (
    analyze_attach_and_segmentation,
    build_osint_normalized_baseline,
    build_sensitivity_matrix,
    estimate_hourly_bar_throughput,
    simulate_linebuster_scenarios,
)
from src.config import load_config
from src.notebook_utils import (
    attach_dilution_stress_test,
    configure_mpl_env,
    date_overlap_audit,
    load_incidents_for_mode,
    pilot_mde_table,
    render_table,
    show_fig,
    split_weekday_weekend_hourly,
    weekday_weekend_peak_table,
)

plt = configure_mpl_env()
PROPOSAL_TARGET_HOURS = (7, 8, 9)
cfg = load_config(repo_root=REPO_ROOT)
DATA_MODE = cfg.data_mode
print(
    f"DATA_MODE={DATA_MODE} | proposal=Mon-Fri {PROPOSAL_TARGET_HOURS} | "
    f"linebuster_start_date={cfg.linebuster_start_date} | tablet_interpretation={cfg.tablet_channel_interpretation}"
)

## 2. OSINT-Normalized Baseline, POS Channel Audit & Peak Bottleneck Decomposition

**Objective:** Exclude nearby OSINT incident days (`<= max_dist`), audit `2026-09-05` channel segmentation (`seg`), and separate weekday vs. weekend peak-hour capacity ceilings.

**Inputs / Assumptions:** `max_dist` selects the smallest configured geofence radius `>= 1500m` (fallback `1500.0`).

**How to Read the Output:** Inspect clean-day revenue, `pos_channel_change` attach metrics, and weekday vs. weekend `P90`/`P95` peak throughput.

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import (
        load_contextual_split,
        load_daily_sales,
        load_hourly_sales,
        load_product_mix,
        load_sales_channels,
    )
    if not cfg.master_sales_db_path.is_file():
        raise FileNotFoundError(f"Missing raw SQLite DB: {cfg.master_sales_db_path}")
    daily_df = load_daily_sales(config=cfg)
    hourly_df = load_hourly_sales(config=cfg, complete_grid=True)
    mix_df = load_product_mix(config=cfg)
    ctx_df = load_contextual_split(config=cfg)
    ch_df = load_sales_channels(config=cfg)
    if daily_df.empty or hourly_df.empty:
        raise ValueError("POS daily/hourly tables returned zero rows in raw mode.")
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])
    hourly_df = pd.read_csv(cfg.sample_dir / "hourly_sales.csv", parse_dates=["date"])
    mix_df = pd.read_csv(cfg.sample_dir / "product_mix.csv", parse_dates=["date"])
    ctx_df = None
    ch_df = None

incidents_df, incidents_source = load_incidents_for_mode(cfg, DATA_MODE, relevant_only=True)
overlap_tbl = date_overlap_audit(daily_df, incidents_df)
render_table(overlap_tbl, title=f"OSINT Window Overlap ({incidents_source.name})")

max_dist = float(min((r for r in cfg.geofence_radii_m if r >= 1500), default=1500.0))
baseline_all = build_osint_normalized_baseline(
    daily_df=daily_df,
    hourly_df=hourly_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    max_distance_m=max_dist,
    peak_hours=cfg.peak_hours,
    operating_hours=cfg.operating_hours,
    weekdays_only=False,
)
baseline = build_osint_normalized_baseline(
    daily_df=daily_df,
    hourly_df=hourly_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    max_distance_m=max_dist,
    peak_hours=PROPOSAL_TARGET_HOURS,
    operating_hours=cfg.operating_hours,
    weekdays_only=True,
)
seg = analyze_attach_and_segmentation(
    daily_df=daily_df,
    product_mix_df=mix_df,
    contextual_split_df=ctx_df,
    sales_channels_df=ch_df,
    tablet_start_date=cfg.tablet_channel_start_date,
    tablet_interpretation=cfg.tablet_channel_interpretation,
    items_end_date=cfg.items_coverage_end_date,
)

scope_comp = pd.DataFrame([
    {"scope": "Full Store (Mon-Sun, cfg.peak_hours)", "clean_days": baseline_all["clean_days"], "clean_mean_daily_rev": baseline_all["clean_mean_daily_revenue"], "target_window_rev": baseline_all["clean_mean_daily_peak_revenue"], "window_share": baseline_all["peak_revenue_share"], "p95_ceiling": baseline_all["peak_p95_daily_ceiling"], "monthly_days": 30.0, "labor_hrs_day": len(cfg.peak_hours)},
    {"scope": "Scoped Proposal (Mon-Fri 07:00-10:00)", "clean_days": baseline["clean_days"], "clean_mean_daily_rev": baseline["clean_mean_daily_revenue"], "target_window_rev": baseline["clean_mean_daily_peak_revenue"], "window_share": baseline["peak_revenue_share"], "p95_ceiling": baseline["peak_p95_daily_ceiling"], "monthly_days": 21.7, "labor_hrs_day": len(PROPOSAL_TARGET_HOURS)},
])
render_table(
    scope_comp,
    title="OSINT-Normalized Baseline: Full Store vs Mon-Fri 7:00-10:00 AM Scope",
    fmt={"clean_mean_daily_rev": "{:,.2f}", "target_window_rev": "{:,.2f}", "window_share": "{:.1%}", "p95_ceiling": "{:,.2f}", "monthly_days": "{:.1f}"},
)

seg_rows = [{"metric": k, "value": f"{v:.4f}" if isinstance(v, float) else str(v)} for k, v in seg.items()]
render_table(pd.DataFrame(seg_rows), title="POS Channel & Attachment Audit (2026-09-05 Guardrail)", max_rows=30)

bar_throughput = estimate_hourly_bar_throughput(
    hourly_df=baseline["clean_hourly_df"],
    daily_df=baseline["clean_daily_df"],
    seg=seg,
    peak_hours=PROPOSAL_TARGET_HOURS,
    operating_hours=cfg.operating_hours,
    weekdays_only=True,
)
render_table(
    bar_throughput[bar_throughput["is_peak"]][["hour_label", "n_days", "mean_volume", "p95_volume", "implied_mean_tickets_hr", "implied_p95_tickets_hr", "implied_mean_drinks_hr", "implied_p95_drinks_hr"]],
    title="Mon-Fri 7:00-10:00 AM Espresso Bar Production Throughput (Tickets/hr & Drinks/hr)",
    fmt={"mean_volume": "{:,.2f}", "p95_volume": "{:,.2f}", "implied_mean_tickets_hr": "{:.2f}", "implied_p95_tickets_hr": "{:.2f}", "implied_mean_drinks_hr": "{:.2f}", "implied_p95_drinks_hr": "{:.2f}"},
)

ww_peak_tbl = weekday_weekend_peak_table(baseline_all["clean_hourly_df"], peak_hours=PROPOSAL_TARGET_HOURS)
render_table(
    ww_peak_tbl[["day_type", "hour_label", "n_days", "mean_volume", "p90_volume", "p95_volume"]],
    title="Morning Window (07:00-09:59): Weekday vs Weekend Profile",
    fmt={"mean_volume": "{:,.2f}", "p90_volume": "{:,.2f}", "p95_volume": "{:,.2f}"},
)

## 3. Ex-Ante 3-Scenario Simulation, Sensitivity Matrix & Attach Dilution Stress Test

**Objective:** Evaluate peak-only Conservative/Base/Optimistic ROI scenarios, gross-margin x peak-lift sensitivity, and downside attach-rate dilution (`-5%`, `0%`, `+5%`).

**Inputs / Assumptions:** Uses `src.causal_eval.simulate_linebuster_scenarios` and `build_sensitivity_matrix`; off-peak revenue stays fixed.

**How to Read the Output:** Check break-even tickets/day, monthly net incremental profit, payback period, and whether Base peak lift survives `-5%` attach dilution.

In [ ]:
scenarios_df = simulate_linebuster_scenarios(baseline=baseline, config=cfg)
render_table(
    scenarios_df[[
        "scenario", "peak_lift_pct", "attach_lift_pct", "total_incremental_rev_day",
        "monthly_net_incremental_profit", "breakeven_tickets_per_day", "payback_months", "pct_of_p95_peak_ceiling",
    ]],
    title="Mon-Fri 7:00-10:00 AM Linebuster ROI Business Case (21.7 weekdays/mo, 3 hrs/day)",
    fmt={"peak_lift_pct": "{:.0%}", "attach_lift_pct": "{:.0%}", "total_incremental_rev_day": "{:,.2f}", "monthly_net_incremental_profit": "{:,.2f}", "breakeven_tickets_per_day": "{:.2f}", "payback_months": "{:.2f}", "pct_of_p95_peak_ceiling": "{:.1%}"},
)

sens_df = build_sensitivity_matrix(baseline=baseline, config=cfg)
pivot = sens_df.pivot(index="gross_margin_pct", columns="peak_lift_pct", values="monthly_net_incremental_profit")
pivot_disp = pivot.copy()
pivot_disp.index = [f"{m:.0%}" for m in pivot_disp.index]
pivot_disp.columns = [f"{l:.0%}" for l in pivot_disp.columns]
pivot_disp = pivot_disp.reset_index().rename(columns={"index": "gross_margin_pct"})
render_table(
    pivot_disp,
    title=f"Mon-Fri 7-10 AM Monthly Net Profit Sensitivity ({cfg.currency})",
    fmt={c: "{:,.2f}" for c in pivot_disp.columns if c != "gross_margin_pct"},
)

stress_df = attach_dilution_stress_test(baseline=baseline, cfg=cfg, attach_lifts=(-0.05, 0.0, 0.05))
render_table(
    stress_df[["scenario", "peak_lift_pct", "attach_lift_pct", "total_incremental_rev_day", "monthly_net_incremental_profit", "payback_months"]],
    title="Mon-Fri 7-10 AM Attach-Rate Dilution Stress Test (Base +10% Lift)",
    fmt={"peak_lift_pct": "{:.0%}", "attach_lift_pct": "{:+.0%}", "total_incremental_rev_day": "{:,.2f}", "monthly_net_incremental_profit": "{:,.2f}", "payback_months": "{:.2f}"},
)

ww_all = split_weekday_weekend_hourly(baseline_all["clean_hourly_df"], peak_hours=PROPOSAL_TARGET_HOURS)
fig, axes = plt.subplots(2, 2, figsize=(13.5, 9.0))

for dtype, col in [("Weekday", "#3288bd"), ("Weekend", "#d95f02")]:
    sub = ww_all[ww_all["day_type"] == dtype]
    axes[0, 0].plot(sub["hour_int"], sub["mean_volume"], marker="o", color=col, label=f"{dtype} Mean")
    axes[0, 0].plot(sub["hour_int"], sub["p95_volume"], linestyle="--", color=col, alpha=0.65, label=f"{dtype} P95")
axes[0, 0].axvspan(6.5, 9.5, color="#fef0d9", alpha=0.75, label="Proposal Window (7-10 AM)")
axes[0, 0].set_title("Clean Hourly Demand: Weekday vs Weekend (Mean & P95)")
axes[0, 0].set_xlabel("Hour of Day")
axes[0, 0].set_ylabel(cfg.currency)
axes[0, 0].legend(fontsize=8)

axes[0, 1].bar(scenarios_df["scenario"], scenarios_df["monthly_net_incremental_profit"], color=["#66c2a5", "#3288bd", "#5e4fa2"])
axes[0, 1].axhline(0.0, color="black", linestyle="--", linewidth=0.9)
axes[0, 1].set_title(f"Mon-Fri 7-10 AM Monthly Net Profit ({cfg.currency})")
axes[0, 1].set_ylabel(cfg.currency)

mat = pivot.to_numpy(dtype=float)
im = axes[1, 0].imshow(mat, cmap="RdYlGn", aspect="auto")
axes[1, 0].set_xticks(range(len(pivot.columns)))
axes[1, 0].set_xticklabels([f"{c:.0%}" for c in pivot.columns])
axes[1, 0].set_yticks(range(len(pivot.index)))
axes[1, 0].set_yticklabels([f"{r:.0%}" for r in pivot.index])
axes[1, 0].set_xlabel("Peak Lift (%)")
axes[1, 0].set_ylabel("Gross Margin (%)")
axes[1, 0].set_title(f"Mon-Fri 7-10 AM Sensitivity Heatmap ({cfg.currency}/mo)")
for r_i in range(mat.shape[0]):
    for c_i in range(mat.shape[1]):
        axes[1, 0].text(c_i, r_i, f"{mat[r_i, c_i]:,.0f}", ha="center", va="center", fontsize=7.5, color="black")
fig.colorbar(im, ax=axes[1, 0], fraction=0.046, pad=0.04)

axes[1, 1].bar(stress_df["scenario"], stress_df["monthly_net_incremental_profit"], color=["#d53e4f", "#fdae61", "#1b9e77"])
axes[1, 1].axhline(0.0, color="black", linestyle="--", linewidth=0.9)
axes[1, 1].set_title(f"Mon-Fri 7-10 AM Attach-Rate Dilution Stress ({cfg.currency}/mo)")
axes[1, 1].set_ylabel(cfg.currency)

show_fig(fig)

## 4. Pre-Registered Pilot Power / MDE Sizing & Causal Readiness

**Objective:** Compute Minimum Detectable Effect (`MDE%`) across pilot horizons `T in {14, 28, 42, 56, 84}` days (`alpha=0.05`, `80%` power) and verify activation gates for post-launch Difference-in-Differences (`estimate_did`).

**Inputs / Assumptions:** Uses observed clean-day daily and peak-hour volatility (`std`); two-sided normal power approximation `(1.96 + 0.84) * sigma * sqrt(2 / T) / mu`.

**How to Read the Output:** Choose pilot duration `T` where `peak_mde_pct` falls below target `+10%` Base lift before switching `LINEBUSTER_START_DATE`.

In [ ]:
mde_tbl = pilot_mde_table(
    clean_daily_df=baseline["clean_daily_df"],
    clean_hourly_df=baseline["clean_hourly_df"],
    peak_hours=PROPOSAL_TARGET_HOURS,
    horizons_days=(14, 28, 42, 56, 84),
    weekdays_only=True,
)
render_table(
    mde_tbl,
    title="Pre-Registered Mon-Fri 7:00-10:00 AM Pilot MDE Sizing (alpha=0.05, 80% Power)",
    fmt={"calendar_weeks": "{:.1f}", "daily_mde_abs": "{:,.2f}", "daily_mde_pct": "{:.1%}", "peak_mde_abs": "{:,.2f}", "peak_mde_pct": "{:.1%}"},
)

readiness_status = "ACTIVE PILOT" if cfg.linebuster_start_date else "EX-ANTE (LINEBUSTER_START_DATE=None)"
print(
    f"Pilot Readiness={readiness_status} | DiD Treatment=Mon-Fri {PROPOSAL_TARGET_HOURS} vs Control=Mon-Fri Off-Peak (11-16) / Weekends"
)